# A1 · OLS with the normal equation and with the SVD

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Implement OLS regression in two ways, with the normal equation and with the SVD decomposition, and find out at
what point the two methods stop giving the same result.

## Dataset

Two synthetic datasets, built in the setup cell:

- `X`, `y`: 100 rows, 3 independent features and known true coefficients (`y = 3 + 2·x₁ − x₂ + 0.5·x₃ + noise`);
- `x_poly`, `y_poly`: 60 points in [0, 1] of a sinusoid with noise, for a polynomial regression. The columns
  1, x, x², x³... of a Vandermonde matrix are nearly collinear: the problem is **ill-conditioned**.

## Educational purpose

Understand why the textbook formula, β = (AᵀA)⁻¹Aᵀy, is not the algorithm used in practice, and what it
concretely means for a matrix to be ill-conditioned.

## Task

1. Write `ols_normal(X, y)`: it adds the column of ones to X, solves the normal equation with `np.linalg.solve` (not with the explicit inverse) and returns the pair `(intercept, coef)`.
2. Write `ols_svd(X, y)`: it computes `U, S, Vt = np.linalg.svd(A, full_matrices=False)` and rebuilds β = V · diag(1/S) · Uᵀ · y, zeroing the singular values smaller than `rcond * S.max()`.
3. On the first dataset check that both functions match `LinearRegression`.
4. On the second dataset, for degrees 4, 6, 8 and 10, build the Vandermonde matrix `A` and fill the DataFrame `table` with the columns `degree`, `cond_A`, `cond_AtA`, `err_normal`, `err_svd`. The error is the maximum absolute difference of the coefficients from the reference `np.linalg.lstsq(A, y_poly, rcond=None)[0]`.
5. Look at the table and assign to `critical_degree` the first degree at which `err_normal` exceeds 1e-3.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
X = rng.normal(size=(100, 3))
y = 3 + X @ np.array([2.0, -1.0, 0.5]) + rng.normal(scale=0.5, size=100)

x_poly = np.linspace(0, 1, 60)
y_poly = np.sin(2 * np.pi * x_poly) + rng.normal(scale=0.1, size=60)

## Your work

In [ ]:
def ols_normal(X, y):
    """Returns (intercept, coef) by solving the normal equation."""
    A = ...            # X with a column of ones in first position
    beta = ...         # np.linalg.solve(A.T @ A, A.T @ y)
    return beta[0], beta[1:]


def ols_svd(X, y, rcond=1e-12):
    """Returns (intercept, coef) using the SVD of A."""
    ...

In [ ]:
rows = []
for degree in [4, 6, 8, 10]:
    A = np.vander(x_poly, degree + 1, increasing=True)        # columns: 1, x, x^2, ..., x^degree
    reference = np.linalg.lstsq(A, y_poly, rcond=None)[0]
    ...
    rows.append({"degree": degree, "cond_A": ..., "cond_AtA": ..., "err_normal": ..., "err_svd": ...})

table = pd.DataFrame(rows)
table

In [ ]:
critical_degree = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For the SVD: `S_inv = np.where(S > rcond * S.max(), 1 / S, 0.0)` and then `beta = Vt.T @ (S_inv * (U.T @ y))`. For the table: the condition number is `np.linalg.cond(matrix)`; to use your functions on the Vandermonde matrix pass `A[:, 1:]` (without the column of ones: they take care of adding it) and recompose `np.r_[intercept, coef]`.'):
    ref = LinearRegression().fit(X, y)
    i_n, c_n = ols_normal(X, y)
    i_s, c_s = ols_svd(X, y)
    check(close(i_n, ref.intercept_) and close(c_n, ref.coef_),
          "ols_normal matches LinearRegression",
          "ols_normal does not match LinearRegression: check the column of ones and the normal equation")
    check(close(i_s, ref.intercept_) and close(c_s, ref.coef_),
          "ols_svd matches LinearRegression",
          "ols_svd does not match LinearRegression: check 1/S and the reconstruction of beta")

    check(list(table.columns) == ["degree", "cond_A", "cond_AtA", "err_normal", "err_svd"],
          "the table has the required columns", "the columns of `table` are not the required ones")
    check((table.err_svd < 1e-6).all(),
          "the SVD stays accurate at every degree", "err_svd should stay below 1e-6 at all degrees")
    check(table.err_normal.is_monotonic_increasing,
          "the error of the normal equation grows with the degree", "err_normal should grow with the degree")
    ratio = np.log10(table.cond_AtA) / np.log10(table.cond_A)
    check(np.allclose(ratio, 2, atol=0.25),
          "cond(AtA) is about cond(A) squared",
          "cond_AtA should be about the square of cond_A: did you compute np.linalg.cond(A.T @ A)?")
    check(critical_degree == 8, "correct critical degree",
          "look at which row of `table` has err_normal exceeding 1e-3 for the first time")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_ols_normal_equation_and_svd.ipynb`. Open it after you pass the check, or after you have really tried.